# Regularized Regression and Model Selection

Lecture 3 gave us a unique least-squares solution when the feature columns are
linearly independent. This demo asks a more practical question: does a unique
coefficient vector remain trustworthy when two features contain almost the
same information?

We will:

1. split the `Credit` customers into two samples and compare the resulting
   coefficients and predictions;
2. use ridge to stabilize weakly identified coefficients;
3. use the lasso when we also want exact zeros; and
4. use `Hitters` to choose the penalty by cross-validation before evaluating
   one final time on a test set.

In [ ]:
# Colab opens this notebook on its own, so fetch the data files that sit
# beside it in the course repository. Running locally, this does nothing.
from pathlib import Path as _Path
import urllib.request as _req
_RAW = 'https://raw.githubusercontent.com/ZhuoranYang/sds265-fall26/main/demos/04-regularized-regression/'
for _name in ['Credit.csv', 'Hitters.csv']:
    if not _Path(_name).exists():
        _req.urlretrieve(_RAW + _name, _name)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 26505
rng = np.random.default_rng(SEED)

credit = pd.read_csv("Credit.csv")
hitters = pd.read_csv("Hitters.csv").dropna().reset_index(drop=True)
credit.head()

## 1. Why collinearity motivates regularization

We begin with a failure mode of ordinary least squares. In `Credit`, a
customer's credit limit and credit rating measure nearly the same underlying
information. A regression can therefore use either feature to make a similar
prediction, making it difficult to decide how much coefficient weight belongs
to each one.

The next two cells make that problem visible. First, we measure the
`Limit`--`Rating` correlation and the condition number of the standardized
design. Then we fit the same six-feature regression separately to customers
1--200 and 201--400. Compare how much the two coefficient vectors change with
how much their predictions change.

In [ ]:
credit_features = ["Income", "Limit", "Rating", "Cards", "Age", "Education"]
X_credit_raw = credit[credit_features].to_numpy(float)
y_credit_raw = credit["Balance"].to_numpy(float)

def fit_standardizer(X):
    mean = X.mean(axis=0)
    scale = X.std(axis=0)
    return mean, scale

def apply_standardizer(X, mean, scale):
    return (X - mean) / scale

mean_credit, scale_credit = fit_standardizer(X_credit_raw)
X_credit = apply_standardizer(X_credit_raw, mean_credit, scale_credit)
y_credit = y_credit_raw - y_credit_raw.mean()

print("Limit/Rating correlation:", credit[["Limit", "Rating"]].corr().iloc[0, 1])
print("condition number of standardized X'X:", np.linalg.cond(X_credit.T @ X_credit))

points = plt.scatter(credit["Limit"], credit["Rating"], c=credit["Balance"],
                     cmap="viridis", s=20, alpha=.75)
plt.colorbar(points, label="balance ($)")
plt.xlabel("credit limit ($)"); plt.ylabel("credit rating");

In [ ]:
# Fit the same six-variable model on two disjoint halves of the customers.
from sklearn.linear_model import LinearRegression

first, second = np.arange(200), np.arange(200, 400)
fit_first = LinearRegression().fit(X_credit_raw[first], y_credit_raw[first])
fit_second = LinearRegression().fit(X_credit_raw[second], y_credit_raw[second])
pred_first = fit_first.predict(X_credit_raw)
pred_second = fit_second.predict(X_credit_raw)

print("prediction correlation:", round(np.corrcoef(pred_first, pred_second)[0, 1], 3))
print("mean absolute prediction gap: $", round(np.mean(np.abs(pred_first - pred_second))))
pd.DataFrame({"first half": fit_first.coef_, "second half": fit_second.coef_},
             index=credit_features).round(2)

The result is the motivation for regularization. The two fits assign very
different weights to `Limit` and `Rating`, yet their predictions have
correlation $0.997$ and differ by only about 33 dollars on average. Least squares
has found a unique answer in each sample, but the answer changes substantially
when the sample changes. We want a fitting rule that is less willing to move
along this weakly identified direction.

### Coding note: `np.linalg.lstsq` versus `np.linalg.solve`

In a from-scratch statistical implementation, we build the design matrix and
derive the equation that defines the estimator; we do **not** need to reimplement
the numerical linear-algebra algorithm. NumPy delegates these computations to
well-tested compiled routines.

- [`np.linalg.lstsq(a, b, rcond=None)`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html)
  accepts a possibly rectangular $m\times p$ matrix `a` and finds the vector
  minimizing $\lVert b-a x\rVert_2$. It returns four objects: the solution,
  residual sums of squares (possibly an empty array), numerical rank, and
  singular values. Thus
  `np.linalg.lstsq(X, y, rcond=None)[0]` means “take the coefficient vector,” not
  “take the first coefficient.” `rcond=None` asks NumPy to use its default
  tolerance when deciding whether a singular value is numerically zero.
- [`np.linalg.solve(a, b)`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html)
  instead solves the square, full-rank system $a x=b$. For $\lambda>0$, ridge
  gives us exactly such a system:
  $(X^\top X+\lambda I)\beta=X^\top y$. We therefore use
  `solve` for ridge, not `lstsq` on the original regression design.

We also prefer `solve(A, b)` to writing `inv(A) @ b`: it states the computational
task directly and avoids explicitly constructing a matrix inverse. The next
cell unpacks all four `lstsq` outputs once; later cells use `[0]` when only the
coefficients are needed.

In [ ]:
least_squares_result = np.linalg.lstsq(X_credit, y_credit, rcond=None)
beta_ls, residual_sums, numerical_rank, singular_values = least_squares_result

print("coefficient shape:", beta_ls.shape)
print("residual sum of squares:", residual_sums.round(1))
print("numerical rank:", numerical_rank, "of", X_credit.shape[1])
print("largest/smallest singular value:",
      singular_values[[0, -1]].round(3))

## 2. Ridge regression from scratch

After centering the response and standardizing every feature, ridge minimizes

$$\frac12\lVert y-X\beta\rVert_2^2 +
\frac{\lambda}{2}\lVert\beta\rVert_2^2.$$

The penalty makes large coefficients costly, especially in directions that the
data identify poorly. The first cell implements the ridge solution directly and
increases $\lambda$ from zero. Watch the total coefficient norm and the
`Limit`--`Rating` pair shrink. The following cell repeats the familiar two-half
comparison from Section 1 and asks whether ridge makes those coefficients less
sensitive to which half of the customers we fit.

In [ ]:
def ridge_closed_form(X, y, alpha):
    # Minimizes 0.5*||y-Xb||^2 + 0.5*alpha*||b||^2.
    system_matrix = X.T @ X + alpha * np.eye(X.shape[1])
    right_hand_side = X.T @ y
    return np.linalg.solve(system_matrix, right_hand_side)

for alpha in [0, 10, 80, 400]:
    if alpha == 0:
        beta = beta_ls
    else:
        beta = ridge_closed_form(X_credit, y_credit, alpha)
    print(f"alpha={alpha:>3}: ||beta||={np.linalg.norm(beta):8.3f}, "
          f"Limit={beta[1]:8.3f}, Rating={beta[2]:8.3f}")

In [ ]:
# Use the same two halves as before. Alpha=40 keeps alpha/n equal to the
# full-data choice 80/400.
half_results = []
for sample_name, index in [("first half", first), ("second half", second)]:
    X_half_raw, y_half_raw = X_credit_raw[index], y_credit_raw[index]
    X_half = (X_half_raw - X_half_raw.mean(axis=0)) / X_half_raw.std(axis=0)
    y_half = y_half_raw - y_half_raw.mean()
    beta_half_ls = np.linalg.lstsq(X_half, y_half, rcond=None)[0]
    beta_half_ridge = ridge_closed_form(X_half, y_half, alpha=40)
    for method, beta in [("least squares", beta_half_ls), ("ridge", beta_half_ridge)]:
        half_results.append({"method": method, "sample": sample_name,
                             **dict(zip(credit_features, beta))})

half_comparison = pd.DataFrame(half_results).set_index(["method", "sample"])
display(half_comparison[["Limit", "Rating"]].round(1))
for method in ["least squares", "ridge"]:
    first_pair = half_comparison.loc[(method, "first half"), ["Limit", "Rating"]]
    second_pair = half_comparison.loc[(method, "second half"), ["Limit", "Rating"]]
    print(f"{method:13s} pair distance: {np.linalg.norm(first_pair-second_pair):.1f}")

Across the two halves, the distance between the standardized
`Limit`--`Rating` coefficient pairs falls from about $407$ for least squares to
$22$ for ridge. This uses only the data split students have already seen: ridge
is less willing to exchange a large amount of weight between two nearly
interchangeable features. Later, validation error will tell us whether that
stability also helps prediction.

### The scikit-learn `Ridge` estimator

For routine modeling we use
[`sklearn.linear_model.Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html).
The common scikit-learn workflow is:

1. construct an estimator, such as
   `ridge = Ridge(alpha=80, fit_intercept=False)`;
2. fit it with `ridge.fit(X, y)`;
3. inspect the learned coefficients in `ridge.coef_`; and
4. predict new responses with `ridge.predict(X_new)`.

Scikit-learn defines the ridge objective as
$\lVert y-X\beta\rVert_2^2+\texttt{alpha}\lVert\beta\rVert_2^2$,
which has the same minimizer as our objective after multiplying both of our
terms by two. We set `fit_intercept=False` only because `X_credit` and `y_credit`
were already centered. With uncentered data, the usual choice is the default
`fit_intercept=True`. The next cell checks that the library estimator and our
`np.linalg.solve` implementation reach the same coefficients.

In [ ]:
# Verify the direct solution against scikit-learn only after implementing it.
from sklearn.linear_model import Ridge

alpha = 80
direct = ridge_closed_form(X_credit, y_credit, alpha)
sklearn_ridge = Ridge(alpha=alpha, fit_intercept=False).fit(X_credit, y_credit)
print("maximum absolute difference:",
      np.max(np.abs(direct - sklearn_ridge.coef_)))
assert np.allclose(direct, sklearn_ridge.coef_)

## 3. When we also want a shorter model: the lasso

Ridge stabilizes a fit, but it normally leaves every coefficient nonzero. If we
also want a model that uses fewer features, we replace the squared penalty by
$\lambda\lVert\beta\rVert_1$. Its corner at zero can set coefficients exactly
to zero.

We first apply the resulting soft-thresholding rule to four numbers so that its
zero region is visible. We then follow the algorithm used in the Lecture 5
slides: take a gradient step on squared error and apply soft thresholding. This
is proximal gradient. Finally, we compare every fitted coefficient with
scikit-learn's result.

In [ ]:
def soft_threshold(value, threshold):
    return np.sign(value) * np.maximum(np.abs(value) - threshold, 0.0)

z = np.array([-2.0, -0.7, 0.2, 1.4])
print("z:     ", z)
print("S_1(z):", soft_threshold(z, 1.0))

In [ ]:
def lasso_proximal_gradient(X, y, penalty, max_steps=50_000, tolerance=1e-10):
    # Objective: ||y-X beta||^2/(2n) + penalty*||beta||_1.
    n, p = X.shape
    lipschitz_constant = np.linalg.eigvalsh(X.T @ X / n).max()
    step_size = 1 / lipschitz_constant
    beta = np.zeros(p)

    for step in range(max_steps):
        squared_error_gradient = X.T @ (X @ beta - y) / n
        after_gradient_step = beta - step_size * squared_error_gradient
        updated = soft_threshold(after_gradient_step, step_size * penalty)
        if np.max(np.abs(updated - beta)) < tolerance:
            return updated, step + 1, step_size
        beta = updated
    raise RuntimeError("proximal gradient did not converge")

penalty = 20.0
beta_lasso, steps, step_size = lasso_proximal_gradient(X_credit, y_credit, penalty)
pd.Series(beta_lasso, index=credit_features).to_frame("from_scratch").assign(
    iterations=steps, step_size=step_size
)

### The scikit-learn `Lasso` estimator

[`sklearn.linear_model.Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)
uses the same estimator interface:
`lasso = Lasso(alpha=20, fit_intercept=False)`, followed by `.fit(X, y)`,
`.coef_`, and `.predict(X_new)`. Its documented objective is exactly

$$\frac{1}{2n}\lVert y-X\beta\rVert_2^2
  +\texttt{alpha}\lVert\beta\rVert_1,$$

so scikit-learn's `alpha` equals the variable called `penalty` in our proximal
gradient function. `max_iter` is a computational limit and `tol` controls the
convergence test; neither is a statistical tuning parameter. Again,
`fit_intercept=False` is appropriate here because we centered the data
ourselves. The next cell verifies the complete coefficient vector, including
the exact zeros.

In [ ]:
from sklearn.linear_model import Lasso

library_lasso = Lasso(alpha=penalty, fit_intercept=False, max_iter=100_000, tol=1e-10)     .fit(X_credit, y_credit).coef_
comparison = pd.DataFrame({
    "from_scratch": beta_lasso,
    "library": library_lasso,
    "absolute_difference": np.abs(beta_lasso - library_lasso),
}, index=credit_features)
comparison

The two implementations agree, including the exact zeros for `Cards` and
`Education`. Those zeros mean that this penalized fit does not use the features;
they do not prove that the features are scientifically irrelevant. Unlike ridge,
the lasso performs estimation and feature selection in one optimization problem.
For a group of nearly interchangeable features, which member survives can still
change with the sample.

### Coefficient paths across penalty values

A fit at one penalty value shows only one point along the regularization path.
The next cell refits each model over a grid of penalties and plots every
standardized coefficient. This lets us see which coefficients are most affected
by regularization and where a coefficient becomes zero.

Python reserves the word `lambda`, so scikit-learn calls the penalty `alpha`.
Read each panel from left to right as the penalty becomes stronger.

In [ ]:
ridge_path_alphas = np.logspace(-2, 4, 100)
lasso_path_alphas = np.logspace(-2, 2.7, 100)

ridge_path = np.array([
    Ridge(alpha=a, fit_intercept=False).fit(X_credit, y_credit).coef_
    for a in ridge_path_alphas
])
lasso_path = np.array([
    Lasso(alpha=a, fit_intercept=False, max_iter=100_000, tol=1e-8)
    .fit(X_credit, y_credit).coef_
    for a in lasso_path_alphas
])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for j, feature in enumerate(credit_features):
    axes[0].plot(ridge_path_alphas, ridge_path[:, j], label=feature)
    axes[1].plot(lasso_path_alphas, lasso_path[:, j], label=feature)
for ax, name in zip(axes, ["ridge", "lasso"]):
    ax.set_xscale("log")
    ax.set(title=f"{name} coefficient path", xlabel="alpha")
axes[0].set_ylabel("coefficient on standardized feature")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout();

## 4. Choose the penalty by five-fold cross-validation

So far we chose the penalty by hand, but the training data alone do not tell us
which amount of shrinkage will predict new observations best. We now switch to
`Hitters`, where log salary is predicted from 16 season and career statistics.

We hide 25% of the players as a final test set. On the remaining players, the
next cell evaluates a grid of penalties by five-fold cross-validation. Each fold
fits its own standardizer and regression model; keeping both steps inside one
pipeline prevents information from the validation fold from leaking into the
fit. In the curves, compare the dashed minimum with the shaded fold-to-fold
uncertainty.

In [ ]:
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Numeric columns only: the deck uses 16 plain counting statistics, since the
# one-hot encoding added complexity without teaching anything.
numeric = [c for c in hitters.columns if hitters[c].dtype != object and c != "Salary"]
X_hitters = hitters[numeric].astype(float)
y_hitters = np.log(hitters["Salary"].to_numpy(float))
print("features:", len(numeric))
train_index, test_index = train_test_split(
    np.arange(len(hitters)), test_size=0.25, random_state=SEED
)
X_train, X_test = X_hitters.iloc[train_index], X_hitters.iloc[test_index]
y_train, y_test = y_hitters[train_index], y_hitters[test_index]
folds = list(KFold(n_splits=5, shuffle=True, random_state=SEED).split(X_train))

def validation_curve(model_name, alphas):
    fold_errors = []
    for alpha in alphas:
        errors = []
        for fit_index, validation_index in folds:
            estimator = (Ridge(alpha=alpha) if model_name == "ridge"
                         else Lasso(alpha=alpha, max_iter=100_000))
            pipeline = make_pipeline(StandardScaler(), estimator)
            pipeline.fit(X_train.iloc[fit_index], y_train[fit_index])
            prediction = pipeline.predict(X_train.iloc[validation_index])
            errors.append(mean_squared_error(y_train[validation_index], prediction) ** 0.5)
        fold_errors.append(errors)
    fold_errors = np.asarray(fold_errors)
    mean = fold_errors.mean(axis=1)
    se = fold_errors.std(axis=1, ddof=1) / np.sqrt(fold_errors.shape[1])
    minimum = int(np.argmin(mean))
    return pd.DataFrame({"alpha": alphas, "mean_rmse": mean, "se": se}), minimum

ridge_curve, ridge_min = validation_curve("ridge", np.logspace(-2, 3, 48))
lasso_curve, lasso_min = validation_curve("lasso", np.logspace(-4, -0.4, 48))

# ISL 6.2.3: select the penalty with the smallest cross-validation error.
print("ridge:", ridge_curve.iloc[ridge_min].round(4).to_dict())
print("lasso:", lasso_curve.iloc[lasso_min].round(4).to_dict())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, curve, minimum, name in [
    (axes[0], ridge_curve, ridge_min, "ridge"),
    (axes[1], lasso_curve, lasso_min, "lasso"),
]:
    ax.plot(curve.alpha, curve.mean_rmse)
    ax.fill_between(curve.alpha, curve.mean_rmse-curve.se, curve.mean_rmse+curve.se, alpha=.15)
    ax.axvline(curve.alpha.iloc[minimum], linestyle="--", label="chosen: smallest CV error")
    ax.set_xscale("log"); ax.set_title(name); ax.set_xlabel("alpha")
axes[0].set_ylabel("validation RMSE (log salary)")
axes[1].legend();

The dip is shallow: a wide range of penalties gives almost the same error,
and the shaded band shows uncertainty across the five folds. Cross-validation
chooses a value; it does not make nearby values meaningfully different.

## 5. One final test evaluation

Cross-validation selected each penalty without using the test responses. We can
now refit least squares, ridge, and lasso on all training players and evaluate
each model once on the untouched test players. Compare test RMSE for prediction;
compare the lasso's nonzero count separately for sparsity.

In [ ]:
ridge_alpha = ridge_curve.alpha.iloc[ridge_min]
lasso_alpha = lasso_curve.alpha.iloc[lasso_min]
models = {
    "least squares": make_pipeline(StandardScaler(), LinearRegression()),
    "ridge": make_pipeline(StandardScaler(), Ridge(alpha=ridge_alpha)),
    "lasso": make_pipeline(StandardScaler(), Lasso(alpha=lasso_alpha, max_iter=100_000)),
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    rmse = mean_squared_error(y_test, model.predict(X_test)) ** 0.5
    nonzero = np.count_nonzero(model[-1].coef_)
    rows.append({"model": name, "test_rmse": rmse, "nonzero_coefficients": nonzero})

selected = [name for name, keep in zip(numeric, models["lasso"][-1].coef_ != 0) if keep]
print("lasso keeps:", ", ".join(selected))
pd.DataFrame(rows).round(4)

On this split, both ridge and lasso improve on least squares, while the lasso
keeps 7 of the 16 features. This is one test-set realization, not a universal
ranking. Predictive error, coefficient stability, and sparsity answer different
questions, so a single number should not stand in for all three.

## Try It

For each change, predict the result before running the cell.

1. Repeat the train/test split over 50 seeds. How variable is the apparent ranking of least squares, ridge, and lasso?
2. Raise the from-scratch lasso penalty from 20 to 80. Which coefficients reach zero first?
3. Move `StandardScaler()` outside the cross-validation loop. Why is the resulting score no longer an honest simulation of the fitting procedure?
4. Replace log salary by salary. Which observations now dominate squared error?